In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [3]:
# tool creation
@tool
def multiply(a : int, b : int) -> int:
    """Given 2 numbers a and b. This tool returns their product"""
    return a * b

In [4]:
print(multiply.invoke(
    {
        "a" : 10,
        "b" : 56
    }
))

560


In [5]:
# tool binding
llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key = os.getenv("GROK_API_KEY"),
    base_url = "https://api.groq.com/openai/v1",
)

In [ ]:
llm_with_tools = llm.bind_tools(
    [multiply] # we can give any number of tools (Tool binding)
)

In [16]:
result1 = llm_with_tools.invoke("Hi how are you") # But we are not getting any type of schema type which we have bind with the tool because on the type of prompt it decides whether we have to use the tool or not

In [17]:
print(result1)

content='Hello! I’m doing great—thanks for asking. How can I help you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 134, 'total_tokens': 193, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 32, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.330935763, 'prompt_time': 0.006514715, 'completion_time': 0.081531933, 'total_time': 0.088046648}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d6de37e6be', 'id': 'chatcmpl-1b3d7b4a-e051-4271-9d82-8055c34924f7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1308e-d56e-7d61-b0da-cd8d10bc7bd6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 134, 'output_tokens': 59, 'total_tokens': 193, 'input_token_details': {}, 'output_token_details': {'reasoning': 32}}


In [ ]:
result1.content # Content is there

'Hello! I’m doing great—thanks for asking. How can I help you today?'

In [ ]:
result1.tool_calls # But no tool, because on the type of query it decides whether we have to use tool or not

[]

In [21]:
result2 = llm_with_tools.invoke("Can you multiple 3 with 10")

In [22]:
result2.content

''

In [23]:
result2.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 10},
  'id': 'fc_a30610ce-be9c-41a2-ac11-0635b8a91261',
  'type': 'tool_call'}]

In [26]:
result = llm_with_tools.invoke("Can you multiple 3 with 10")

In [27]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'fc_0f800d8d-eb58-4464-8d66-4bc8b1dab4a5',
 'type': 'tool_call'}

In [29]:
multiply.invoke(result.tool_calls[0]["args"])

30

In [ ]:
# Getting Tool Message as print
# We can pass this to the LLM
multiply.invoke(
    {
        'name': 'multiply',
        'args': {'a': 3, 'b': 10},
        'id': 'fc_0f800d8d-eb58-4464-8d66-4bc8b1dab4a5',
        'type': 'tool_call'
    }
)

ToolMessage(content='30', name='multiply', tool_call_id='fc_0f800d8d-eb58-4464-8d66-4bc8b1dab4a5')